# Room Style Classifier — Training & Benchmarking (Colab)

Berre.ca furniture recommender project. Run cells top to bottom.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU (T4).

Includes dropout regularization on the classifier head and early stopping on validation accuracy.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 1. Mount Google Drive and unzip the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Adjust this path to wherever you uploaded data.zip in your Drive
ZIP_PATH = "/content/drive/MyDrive/data.zip"

import zipfile, os
os.makedirs("/content/room_stylist", exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall("/content/room_stylist")

print("Extracted. Contents:")
!ls /content/room_stylist/data

## 2. Install extra dependency for the CLIP zero-shot baseline

In [ ]:
!pip install -q open_clip_torch

## 3. Imports and config

In [ ]:
import time
import json
import copy
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("/content/room_stylist/data")
BATCH_SIZE = 32
MAX_EPOCHS = 20        # upper bound; early stopping will usually cut this short
LR = 3e-4
IMG_SIZE = 224
DROPOUT_P = 0.3         # dropout before the final classification layer
EARLY_STOP_PATIENCE = 3  # stop if val_acc doesn't improve for this many epochs

print("Using device:", DEVICE)

## 4. Data loaders

In [ ]:
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(DATA_DIR / "train", transform=train_tfms)
val_ds = datasets.ImageFolder(DATA_DIR / "val", transform=eval_tfms)
test_ds = datasets.ImageFolder(DATA_DIR / "test", transform=eval_tfms)

class_names = train_ds.classes
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 5. Model builders

Both heads now use `Dropout(DROPOUT_P) -> Linear` instead of a bare `Linear`, to reduce overfitting on a moderately-sized dataset.

In [ ]:
def build_resnet18(num_classes, dropout_p=DROPOUT_P):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(in_features, num_classes),
    )
    return model


def build_efficientnet_b0(num_classes, dropout_p=DROPOUT_P):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    in_features = model.classifier[1].in_features
    # EfficientNet already ships a Dropout before its classifier; replace both
    # so we control the rate explicitly.
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(in_features, num_classes),
    )
    return model

## 6. Train / evaluate helper functions

`train_model` now tracks the best validation accuracy, keeps a copy of the
best-performing weights, and stops early if validation accuracy hasn't
improved for `EARLY_STOP_PATIENCE` consecutive epochs — avoiding wasted
compute and overfitting from training too long.

In [ ]:
def train_model(model, train_loader, val_loader, max_epochs=MAX_EPOCHS, lr=LR,
                 patience=EARLY_STOP_PATIENCE):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=lr)

    best_val_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE)
                out = model(x)
                preds = out.argmax(dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(y.numpy())
        val_acc = accuracy_score(val_true, val_preds)

        improved = val_acc > best_val_acc
        marker = "  <- best so far" if improved else ""
        print(f"  epoch {epoch+1}/{max_epochs}  train_loss={train_loss:.4f}  val_acc={val_acc:.4f}{marker}")

        if improved:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping: no improvement for {patience} epochs "
                      f"(best val_acc={best_val_acc:.4f} at epoch {epoch+1-patience})")
                break

    model.load_state_dict(best_state)
    return model


def evaluate_model(model, test_loader, class_names):
    model.eval()
    preds, true = [], []
    start = time.time()
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            out = model(x)
            batch_preds = out.argmax(dim=1).cpu().numpy()
            preds.extend(batch_preds)
            true.extend(y.numpy())
    elapsed = time.time() - start
    per_image_ms = (elapsed / len(test_loader.dataset)) * 1000

    acc = accuracy_score(true, preds)
    f1 = f1_score(true, preds, average="macro")
    cm = confusion_matrix(true, preds)
    report = classification_report(true, preds, target_names=class_names)

    return {
        "accuracy": acc,
        "macro_f1": f1,
        "inference_ms_per_image": per_image_ms,
        "confusion_matrix": cm.tolist(),
        "report": report,
    }

## 7. Train + benchmark ResNet-18

In [ ]:
print("=== ResNet-18 ===")
resnet = build_resnet18(len(class_names))
resnet = train_model(resnet, train_loader, val_loader)
resnet_results = evaluate_model(resnet, test_loader, class_names)
print(resnet_results["report"])

## 8. Train + benchmark EfficientNet-B0

In [ ]:
print("=== EfficientNet-B0 ===")
effnet = build_efficientnet_b0(len(class_names))
effnet = train_model(effnet, train_loader, val_loader)
effnet_results = evaluate_model(effnet, test_loader, class_names)
print(effnet_results["report"])

## 9. CLIP zero-shot baseline (no training needed)

In [ ]:
import open_clip
from PIL import Image

clip_model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(DEVICE).eval()

# Tune these prompts to improve zero-shot accuracy
prompt_map = {
    "mid_century_modern": "a photo of a mid-century modern style living room",
    "contemporary_scandinavian": "a photo of a contemporary scandinavian style living room",
    "traditional_classic": "a photo of a traditional, ornate, classic living room",
    "rustic_farmhouse": "a photo of a rustic farmhouse style living room",
    "coastal_tropical": "a photo of a coastal or tropical style living room",
    "eclectic_industrial": "a photo of an eclectic industrial style living room",
}
prompts = [prompt_map.get(c, f"a photo of a {c.replace('_', ' ')} living room") for c in class_names]
text_tokens = tokenizer(prompts).to(DEVICE)

with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

# Build a raw-PIL test set (no normalization tensor) for CLIP's own preprocessing
raw_test_ds = datasets.ImageFolder(DATA_DIR / "test")  # default loader returns PIL images

preds, true = [], []
start = time.time()
with torch.no_grad():
    for img, label in raw_test_ds:
        image_input = preprocess(img).unsqueeze(0).to(DEVICE)
        image_features = clip_model.encode_image(image_input)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        sims = (image_features @ text_features.T).squeeze(0)
        pred = sims.argmax().item()
        preds.append(pred)
        true.append(label)
elapsed = time.time() - start
per_image_ms = (elapsed / len(true)) * 1000

clip_results = {
    "accuracy": accuracy_score(true, preds),
    "macro_f1": f1_score(true, preds, average="macro"),
    "inference_ms_per_image": per_image_ms,
}
print(clip_results)

## 10. Summary table + save the winning model

In [ ]:
results = {
    "resnet18": resnet_results,
    "efficientnet_b0": effnet_results,
    "clip_zero_shot": clip_results,
}

print(f"{'Model':20s} {'Accuracy':>10s} {'Macro F1':>10s} {'ms/img':>10s}")
for name, r in results.items():
    print(f"{name:20s} {r['accuracy']:10.4f} {r['macro_f1']:10.4f} {r['inference_ms_per_image']:10.1f}")

# Save results (excluding confusion matrices/reports for a clean JSON)
summary = {k: {kk: vv for kk, vv in v.items() if kk in ("accuracy", "macro_f1", "inference_ms_per_image")}
           for k, v in results.items()}
with open("/content/benchmark_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved benchmark_results.json")

In [ ]:
# Once you've picked the winning model based on the table above,
# save its weights and download them for the Flask app deployment step.

# Example if EfficientNet-B0 wins:
torch.save(effnet.state_dict(), "/content/style_classifier_effnet_b0.pt")

from google.colab import files
files.download("/content/style_classifier_effnet_b0.pt")